# `_state_passing_fwd` in JAX Pallas

## What this kernel does

Given:
- `states (batch, nchunks, nheads, dstate)` — chunk state contributions from `chunk_state_fwd`
- `dA_chunk_cumsum (batch, nheads, nchunks)` — cumulative sum of log-A factors per chunk

It runs a **sequential linear recurrence** over chunks:

```
carry = 0   # (or initial_states if provided)
out[0] = carry
for c in range(nchunks):
    carry = exp(dA_cs[c]) * carry + states[c]
    if c < nchunks-1:
        out[c+1] = carry
    else:
        final_states = carry
```

`out[c]` is the SSM state **before** processing chunk `c`. It serves as the initial hidden
state for each chunk in the backward pass.

## Three implementations

| Implementation | Complexity | Notes |
|---|---|---|
| `lax.scan` (naive) | O(nchunks) serial | Sequential, matches Triton exactly |
| `lax.associative_scan` | O(log nchunks) parallel | Better for large nchunks |
| Pallas kernel | O(nchunks) serial | GPU kernel via Python for-loop unrolled at trace time |

In [1]:
import os, sys, types, math, time
import numpy as np

# Needed so the Triton backend inside JAX can find the right GCC
os.environ.setdefault(
    "CC",
    os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"),
)

import jax
import jax.numpy as jnp
from jax import lax
import jax.experimental.pallas as pl
from jax._src.pallas.triton.core import CompilerParams

# ── Mamba path (for Triton reference only) ──────────────────────────────────
# We create a minimal stub for the mamba_ssm package so we can import just
# ssd_state_passing without triggering __init__.py (which needs CUDA extensions
# like selective_scan_cuda and huggingface_hub that aren't installed here).
MAMBA_ROOT = os.path.expanduser("~/mamba")
pkg = types.ModuleType("mamba_ssm")
pkg.__path__    = [os.path.join(MAMBA_ROOT, "mamba_ssm")]
pkg.__package__ = "mamba_ssm"
sys.modules["mamba_ssm"] = pkg

# ssd_state_passing.py imports mamba_ssm.utils.determinism.autotune_configs
# — mock it with a passthrough so Triton's @autotune decorator still works
utils_mod = types.ModuleType("mamba_ssm.utils")
utils_mod.__path__ = []
det_mod = types.ModuleType("mamba_ssm.utils.determinism")
det_mod.autotune_configs = lambda cfgs: cfgs
sys.modules["mamba_ssm.utils"] = utils_mod
sys.modules["mamba_ssm.utils.determinism"] = det_mod

import torch
from mamba_ssm.ops.triton.ssd_state_passing import _state_passing_fwd

print(f"JAX version : {jax.__version__}")
print(f"JAX devices : {jax.devices()}")
print(f"PyTorch     : {torch.__version__}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")

JAX version : 0.9.0.1
JAX devices : [CudaDevice(id=0)]
PyTorch     : 2.8.0+cu129
GPU         : NVIDIA GeForce RTX 4090


## Triton Reference: `_state_passing_fwd_kernel`

The Triton kernel is launched with a 3-D grid:
```
grid = (cdiv(dstate, BLOCK_SIZE), batch, nheads)
```
Each program tile processes one `(batch, head)` pair and a `BLOCK_SIZE` slice of `dstate`.  
The sequential loop over `nchunks` runs entirely within each program (no inter-block communication needed).

**Key observation:** Each Triton program holds the running `state` vector (`BLOCK_SIZE` floats) in registers across all `nchunks` iterations. This is a trivial sequential scan — no synchronization required.

In [2]:
# ---- Triton kernel (annotated) ----
# Source: mamba/mamba_ssm/ops/triton/ssd_state_passing.py
#
# @triton.jit
# def _state_passing_fwd_kernel(
#     states_ptr, out_ptr, final_states_ptr, dA_cs_ptr, ...
#     dim, nchunks, ...
#     stride_states_batch, stride_states_chunk, stride_states_head, stride_states_dim,
#     ...
#     HAS_INITSTATES: tl.constexpr, HAS_SEQ_IDX: tl.constexpr,
#     BLOCK_SIZE: tl.constexpr,
# ):
#     # ---- Grid: axis-0 = dim tiles, axis-1 = batch, axis-2 = nheads ----
#     pid_b = tl.program_id(axis=1)          # batch index
#     pid_h = tl.program_id(axis=2)          # head index
#     pid_m = tl.program_id(axis=0)          # dstate tile index
#
#     # Advance each pointer to this program's (batch, head) position
#     states_ptr += pid_b * stride_states_batch + pid_h * stride_states_head
#     dA_cs_ptr  += pid_b * stride_dA_cs_batch  + pid_h * stride_dA_cs_head
#     out_ptr    += pid_b * stride_out_batch     + pid_h * stride_out_head
#     final_states_ptr += pid_b * ... + pid_h * ...
#
#     # BLOCK_SIZE contiguous elements in dstate dimension for this tile
#     offs_m = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # shape: (BLOCK_SIZE,)
#     states_ptrs      = states_ptr      + offs_m * stride_states_dim  # pointer per element
#     out_ptrs         = out_ptr         + offs_m * stride_out_dim
#     final_states_ptrs = final_states_ptr + offs_m * stride_final_states_dim
#
#     # Initialize carry (float32 for numerical stability)
#     states = tl.zeros((BLOCK_SIZE,), dtype=tl.float32)  # shape: (BLOCK_SIZE,)
#
#     # ---- Store out[0] = initial state (0 or initial_states) ----
#     tl.store(out_ptrs, states, mask=offs_m < dim)
#     out_ptrs += stride_out_chunk        # advance out pointer to out[1]
#
#     # ---- Sequential recurrence over nchunks ----
#     for c in range(nchunks):
#         # Load states[batch, c, head, pid_m*BLOCK_SIZE : (pid_m+1)*BLOCK_SIZE]
#         new_states = tl.load(states_ptrs, mask=offs_m < dim, other=0.0).to(tl.float32)
#         #                                   shape: (BLOCK_SIZE,)
#
#         # Load dA_chunk_cumsum[batch, head, c]  — a single SCALAR per chunk
#         # (The head stride was already applied; chunk stride advances the pointer)
#         dA_cs = tl.load(dA_cs_ptr).to(tl.float32)   # scalar
#         scale = tl.exp(dA_cs)                         # scalar
#
#         # Recurrence: carry = exp(dA_cs[c]) * carry + states[c]
#         states = scale * states + new_states          # shape: (BLOCK_SIZE,)
#
#         if c < nchunks - 1:
#             tl.store(out_ptrs, states, mask=offs_m < dim)   # out[c+1] = carry
#         else:
#             tl.store(final_states_ptrs, states, mask=offs_m < dim)  # final_states
#
#         states_ptrs += stride_states_chunk  # advance: states[c] -> states[c+1]
#         dA_cs_ptr   += stride_dA_cs_chunk   # advance: dA_cs[c] -> dA_cs[c+1]
#         out_ptrs    += stride_out_chunk     # advance: out[c+1] -> out[c+2]

print("Triton reference noted (see source above)")

Triton reference noted (see source above)


## Naive JAX: `lax.scan`

`jax.lax.scan` maps directly to the sequential recurrence. It runs serially on GPU (no kernel fusion across steps), but is a useful correctness reference.

In [3]:
def state_passing_naive(states, dA_chunk_cumsum):
    """
    Sequential linear recurrence over chunks using jax.lax.scan.

    Matches the Triton kernel exactly: sequential O(nchunks) computation.

    Args:
        states:           (batch, nchunks, nheads, dstate)  float32
        dA_chunk_cumsum:  (batch, nheads, nchunks)          float32

    Returns:
        out:          (batch, nchunks, nheads, dstate)  — state BEFORE each chunk
        final_states: (batch, nheads, dstate)            — state AFTER all chunks
    """
    batch, nchunks, nheads, dstate = states.shape

    # Rearrange dA so chunks are axis-1 (to align with states layout)
    # dA_chunk_cumsum: (batch, nheads, nchunks) -> (batch, nchunks, nheads)
    dA = dA_chunk_cumsum.transpose(0, 2, 1)

    def scan_fn(carry, x):
        """
        carry: (batch, nheads, dstate)          — current SSM state
        x[0]:  (batch, nheads, dstate)          — new state contribution (states[c])
        x[1]:  (batch, nheads)                  — dA_chunk_cumsum[c] (scalar per head)

        Output: updated carry, the state BEFORE this chunk (for storage in 'out').
        """
        new_state, dA_c = x
        out_state = carry                           # state BEFORE this chunk -> output
        scale = jnp.exp(dA_c)[..., None]           # (batch, nheads, 1) — broadcasts over dstate
        next_carry = scale * carry + new_state      # (batch, nheads, dstate)
        return next_carry, out_state

    # lax.scan scans over axis-0, so move nchunks to front
    states_T = states.transpose(1, 0, 2, 3)    # (nchunks, batch, nheads, dstate)
    dA_T     = dA.transpose(1, 0, 2)           # (nchunks, batch, nheads)

    init_carry = jnp.zeros((batch, nheads, dstate), dtype=jnp.float32)
    final_states, out_T = lax.scan(scan_fn, init_carry, (states_T, dA_T))
    # out_T:  (nchunks, batch, nheads, dstate)  — out_T[c] = state BEFORE chunk c

    out = out_T.transpose(1, 0, 2, 3)          # (batch, nchunks, nheads, dstate)
    return out, final_states

print("state_passing_naive defined")

state_passing_naive defined


## Associative Scan: `lax.associative_scan`

The linear recurrence `y[c] = d[c] * y[c-1] + s[c]` (with `y[-1] = 0`) forms a monoid under:

```
(d_l, s_l) ⊕ (d_r, s_r)  =  (d_r * d_l,  d_r * s_l + s_r)
```

where:
- `d[c] = exp(dA_cs[c])` is the multiplicative discount at step c
- `s[c] = states[c]` is the additive contribution at step c
- The product `⊕` represents composing two recurrence "segments"

`lax.associative_scan` computes all prefix products in **O(log nchunks)** parallel steps using the work-efficient Blelloch scan algorithm.

**After the scan:** `cumul[c] = (d[0]*...*d[c], y[c])` where `y[c]` = state after chunk c.  
We shift by 1 to get `out[c]` = state *before* chunk c.

In [4]:
def state_passing_assoc(states, dA_chunk_cumsum):
    """
    Parallel prefix scan via jax.lax.associative_scan.

    O(log nchunks) parallel steps instead of O(nchunks) serial steps.
    More efficient for large nchunks (e.g. 128+) on GPU.

    Args:
        states:           (batch, nchunks, nheads, dstate)  float32
        dA_chunk_cumsum:  (batch, nheads, nchunks)          float32

    Returns:
        out:          (batch, nchunks, nheads, dstate)
        final_states: (batch, nheads, dstate)
    """
    batch, nchunks, nheads, dstate = states.shape

    # discounts[b, c, h] = exp(dA_chunk_cumsum[b, h, c])
    # Shape: (batch, nchunks, nheads)  — rearrange to put nchunks on axis-1
    discounts = jnp.exp(dA_chunk_cumsum.transpose(0, 2, 1))  # (batch, nchunks, nheads)

    def combine(left, right):
        """
        Monoid combine: (d_l, s_l) ⊕ (d_r, s_r) = (d_r * d_l, d_r * s_l + s_r)

        'd' is the accumulated discount, 's' is the accumulated state.
        'right' is later in the sequence, so it scales left's accumulated state.

        left/right shapes (inside associative_scan, axis=1 is the scan axis):
          d: (batch, k, nheads)         — k varies with scan depth
          s: (batch, k, nheads, dstate)
        """
        d_l, s_l = left
        d_r, s_r = right
        # d_r[..., None] broadcasts (batch, k, nheads) -> (batch, k, nheads, 1)
        # so it multiplies across dstate dimension
        return (d_r * d_l, d_r[..., None] * s_l + s_r)

    # Pack elements: each chunk contributes (discount, state_contribution)
    elements = (discounts, states)   # axis-1 is nchunks for both

    # Inclusive prefix scan over the nchunks axis
    # cumul_d[b, c, h]     = exp(dA_cs[b,h,0]) * ... * exp(dA_cs[b,h,c])
    # cumul_s[b, c, h, :]  = y[c] = state AFTER chunk c
    _cumul_d, cumul_s = lax.associative_scan(combine, elements, axis=1)

    # Shift to get 'out': out[c] = state BEFORE chunk c
    #   out[0]   = 0  (no prior state)
    #   out[c]   = cumul_s[c-1]  for c >= 1
    zeros = jnp.zeros((batch, 1, nheads, dstate), dtype=states.dtype)
    out = jnp.concatenate([zeros, cumul_s[:, :-1, :, :]], axis=1)  # (batch, nchunks, nheads, dstate)

    final_states = cumul_s[:, -1, :, :]     # (batch, nheads, dstate) — state after ALL chunks

    return out, final_states

print("state_passing_assoc defined")

state_passing_assoc defined


## Pallas Kernel

### Design

We mirror the Triton kernel layout but collapse `(batch, nheads)` into one grid dimension:

```
Grid: (batch * nheads,  ceil(dstate / BLOCK_SIZE))
         bh              pm
```

Each kernel instance handles one `(bh, pm)` pair, loading the full `nchunks` worth of
`states` and `dA_cs` into a `(nchunks, BLOCK_SIZE)` working set.

### Pallas ref access rules

- `ref[0, c, :]` — two **static Python int** leading indices + full trailing slice → **valid** ✓  
- `ref[0, c]`   — two static scalars, NO trailing slice → generates `lax.squeeze(lax.slice)` → **FAILS** ✗

To avoid the scalar issue, `dA_chunk_cumsum` gets an extra trailing dimension of size 1:
- Shape becomes `(BH, nchunks, 1)` so we can use `dA_ref[0, c, :]` → `(1,)` which broadcasts.

### Loop unrolling

Because we use a **Python for-loop** (not `jax.lax.fori_loop`), JAX unrolls the loop at trace time.
`c` is a concrete Python int in each iteration, so `states_ref[0, c, :]` and `out_ref[0, c, :]`
are static pointer offsets — exactly what Triton's `states_ptrs + c * stride_states_chunk` computes.

In [5]:
def _make_state_passing_kernel(nchunks, BLOCK_SIZE):
    """
    Factory that captures nchunks and BLOCK_SIZE as compile-time constants.
    Returns a Pallas kernel function.
    """

    def _state_passing_fwd_kernel(
        states_ref,   # (1, nchunks, BLOCK_SIZE)  float32
        #   Triton: states_ptr + pid_b*stride_b + pid_h*stride_h
        #           tl.load(states_ptrs + c*stride_chunk, mask=offs_m < dim)

        dA_cs_ref,    # (1, nchunks, 1)           float32  [extra '1' for safe ref access]
        #   Triton: dA_cs_ptr + pid_b*stride_b + pid_h*stride_h
        #           tl.load(dA_cs_ptr + c*stride_chunk)  <- scalar load
        #   Pallas: dA_cs_ref[0, c, :] -> (1,) so we avoid scalar ref indexing

        out_ref,      # (1, nchunks, BLOCK_SIZE)  float32
        #   Triton: out_ptr + pid_b*stride_b + pid_h*stride_h
        #           tl.store(out_ptrs + c*stride_chunk, states)

        final_ref,    # (1, BLOCK_SIZE)            float32
        #   Triton: final_states_ptr + pid_b*stride_b + pid_h*stride_h
        #           tl.store(final_states_ptrs, states)  [only on last chunk]
    ):
        # ---- Triton equivalent: initialize carry ----
        # Triton: states = tl.zeros((BLOCK_SIZE,), dtype=tl.float32)
        state = jnp.zeros((BLOCK_SIZE,), dtype=jnp.float32)

        # ---- Store out[0] = initial state (zeros) ----
        # Triton: tl.store(out_ptrs, states, mask=offs_m < dim)
        #         out_ptrs += stride_out_chunk
        out_ref[0, 0, :] = state      # ref[scalar, scalar, full_slice] ✓

        # ---- Sequential recurrence over all chunks ----
        # Triton: for c in range(nchunks):
        # Python for-loop: unrolled by JAX tracer — c is a static Python int each iteration
        for c in range(nchunks):

            # Load new state contribution for this chunk
            # Triton: new_states = tl.load(states_ptrs, mask=offs_m < dim).to(tl.float32)
            #         states_ptrs += stride_states_chunk
            new_state = states_ref[0, c, :]   # (BLOCK_SIZE,) — ref[scalar, static_c, full_slice] ✓

            # Load dA_chunk_cumsum scalar for this chunk
            # Triton: dA_cs = tl.load(dA_cs_ptr).to(tl.float32)
            #         dA_cs_ptr += stride_dA_cs_chunk
            dA = dA_cs_ref[0, c, :]           # (1,) — ref[scalar, static_c, full_slice] ✓

            # Compute discount factor
            # Triton: scale = tl.exp(dA_cs)
            scale = jnp.exp(dA)               # (1,) — broadcasts to (BLOCK_SIZE,)

            # Recurrence step: carry = exp(dA_cs) * carry + states[c]
            # Triton: states = scale * states + new_states
            state = scale * state + new_state  # (BLOCK_SIZE,)

            if c < nchunks - 1:
                # Store updated state into out[c+1]
                # Triton: tl.store(out_ptrs, states, mask=offs_m < dim)
                #         out_ptrs += stride_out_chunk
                out_ref[0, c + 1, :] = state  # ref[scalar, static_c+1, full_slice] ✓
            else:
                # Last chunk: store to final_states
                # Triton: tl.store(final_states_ptrs, states, mask=offs_m < dim)
                final_ref[0, :] = state        # ref[scalar, full_slice] ✓

    return _state_passing_fwd_kernel

print("_make_state_passing_kernel defined")

_make_state_passing_kernel defined


In [6]:
def state_passing_pallas(states, dA_chunk_cumsum, BLOCK_SIZE=64):
    """
    Pallas implementation of _state_passing_fwd.

    Grid: (batch*nheads, ceil(dstate/BLOCK_SIZE))

    Each kernel instance processes:
      - one (batch, head) pair — all nchunks iterations
      - one BLOCK_SIZE tile along dstate

    Args:
        states:           (batch, nchunks, nheads, dstate)  float32
        dA_chunk_cumsum:  (batch, nheads, nchunks)          float32
        BLOCK_SIZE:       int, tile size along dstate (must divide dstate)

    Returns:
        out:          (batch, nchunks, nheads, dstate)  float32
        final_states: (batch, nheads, dstate)           float32
    """
    batch, nchunks, nheads, dstate = states.shape
    BH = batch * nheads
    n_tiles = dstate // BLOCK_SIZE  # assumes dstate % BLOCK_SIZE == 0

    # ---- Reshape inputs ----
    # Triton uses separate batch/head grid dims (pid_b, pid_h).
    # We collapse them for simplicity.
    #
    # states: (batch, nchunks, nheads, dstate)
    #   -> transpose(0,2,1,3): (batch, nheads, nchunks, dstate)  [makes BH contiguous]
    #   -> reshape: (BH, nchunks, dstate)
    states_flat = states.transpose(0, 2, 1, 3).reshape(BH, nchunks, dstate)

    # dA_chunk_cumsum: (batch, nheads, nchunks)
    #   -> reshape: (BH, nchunks)
    #   -> [..., None]: (BH, nchunks, 1)  — extra dim so ref[0, c, :] gives (1,) not scalar
    #
    # Why the extra dim?
    # Inside the kernel, ref[0, c]  (scalar) -> lax.squeeze(lax.slice(.)) -> NotImplementedError
    # But ref[0, c, :] (full trailing slice) -> Pallas load primitive -> OK
    dA_flat = dA_chunk_cumsum.reshape(BH, nchunks)[:, :, None]   # (BH, nchunks, 1)

    # ---- BlockSpecs ----
    # Grid index (bh, pm) -> block covering:
    #   states_flat[bh, 0:nchunks, pm*BLOCK_SIZE:(pm+1)*BLOCK_SIZE]
    #   dA_flat   [bh, 0:nchunks, 0:1]
    #   out_flat  [bh, 0:nchunks, pm*BLOCK_SIZE:(pm+1)*BLOCK_SIZE]
    #   final_flat[bh, pm*BLOCK_SIZE:(pm+1)*BLOCK_SIZE]
    #
    # BlockSpec((block_shape), index_map): index_map returns BLOCK-level indices
    # (each return value i is multiplied by block_shape[dim] to get the element offset)

    in_specs = [
        # states_flat: (BH, nchunks, dstate)
        # Block (1, nchunks, BLOCK_SIZE): load all nchunks at once, tile over dstate
        pl.BlockSpec((1, nchunks, BLOCK_SIZE), lambda bh, pm: (bh, 0, pm)),

        # dA_flat: (BH, nchunks, 1)
        # Block (1, nchunks, 1): load all nchunks at once; last dim always 0
        # 'pm' (dstate tile) doesn't affect dA — same values for all tiles
        pl.BlockSpec((1, nchunks, 1), lambda bh, pm: (bh, 0, 0)),
    ]

    out_specs = [
        # out_flat: (BH, nchunks, dstate) — same layout as states_flat
        pl.BlockSpec((1, nchunks, BLOCK_SIZE), lambda bh, pm: (bh, 0, pm)),

        # final_flat: (BH, dstate) — one tile of BLOCK_SIZE per (bh, pm)
        pl.BlockSpec((1, BLOCK_SIZE), lambda bh, pm: (bh, pm)),
    ]

    # ---- Build and call the kernel ----
    kernel = _make_state_passing_kernel(nchunks, BLOCK_SIZE)

    out_flat, final_flat = pl.pallas_call(
        kernel,
        out_shape=[
            jax.ShapeDtypeStruct((BH, nchunks, dstate), jnp.float32),  # out
            jax.ShapeDtypeStruct((BH, dstate),           jnp.float32),  # final_states
        ],
        in_specs=in_specs,
        out_specs=out_specs,
        grid=(BH, n_tiles),
        compiler_params=CompilerParams(),   # force Triton backend (not mosaic_gpu)
    )(states_flat, dA_flat)

    # ---- Reshape outputs back to original layout ----
    # out_flat: (BH, nchunks, dstate)
    #   -> reshape: (batch, nheads, nchunks, dstate)
    #   -> transpose(0,2,1,3): (batch, nchunks, nheads, dstate)
    out = out_flat.reshape(batch, nheads, nchunks, dstate).transpose(0, 2, 1, 3)

    # final_flat: (BH, dstate) -> (batch, nheads, dstate)
    final_states = final_flat.reshape(batch, nheads, dstate)

    return out, final_states

print("state_passing_pallas defined")

state_passing_pallas defined


## Correctness Check

Compare all three JAX implementations against the Triton reference.

In [7]:
# ---- Test shapes ----
batch, nchunks, nheads, dstate = 2, 8, 4, 64
key = jax.random.PRNGKey(42)

# Random inputs — float32 throughout
states_jax = jax.random.normal(key, (batch, nchunks, nheads, dstate), dtype=jnp.float32)
dA_jax     = jax.random.normal(jax.random.PRNGKey(1),
                                (batch, nheads, nchunks), dtype=jnp.float32) * 0.1
# Small dA values (log-scale), so exp(dA) is near 1 — numerically stable

# ---- Triton reference ----
states_torch = torch.from_numpy(np.array(states_jax)).cuda().float()
dA_torch     = torch.from_numpy(np.array(dA_jax)).cuda().float()
out_triton, final_triton = _state_passing_fwd(states_torch, dA_torch)

out_ref   = jnp.array(out_triton.cpu().numpy())    # (batch, nchunks, nheads, dstate)
final_ref = jnp.array(final_triton.cpu().numpy())  # (batch, nheads, dstate)

print(f"out shape:          {out_ref.shape}")
print(f"final_states shape: {final_ref.shape}")
print()

# ---- Naive (lax.scan) ----
out_naive, final_naive = jax.jit(state_passing_naive)(states_jax, dA_jax)
print(f"Naive vs Triton — out max diff:          {jnp.max(jnp.abs(out_naive - out_ref)):.2e}")
print(f"Naive vs Triton — final_states max diff: {jnp.max(jnp.abs(final_naive - final_ref)):.2e}")

# ---- Associative scan ----
out_assoc, final_assoc = jax.jit(state_passing_assoc)(states_jax, dA_jax)
print(f"Assoc vs Triton — out max diff:          {jnp.max(jnp.abs(out_assoc - out_ref)):.2e}")
print(f"Assoc vs Triton — final_states max diff: {jnp.max(jnp.abs(final_assoc - final_ref)):.2e}")

# ---- Pallas ----
out_pallas, final_pallas = jax.jit(state_passing_pallas)(states_jax, dA_jax)
print(f"Pallas vs Triton — out max diff:          {jnp.max(jnp.abs(out_pallas - out_ref)):.2e}")
print(f"Pallas vs Triton — final_states max diff: {jnp.max(jnp.abs(final_pallas - final_ref)):.2e}")
print()
print("All differences should be < 1e-4 for float32 arithmetic")

out shape:          (2, 8, 4, 64)
final_states shape: (2, 4, 64)

Naive vs Triton — out max diff:          9.54e-07
Naive vs Triton — final_states max diff: 1.91e-06
Assoc vs Triton — out max diff:          1.91e-06
Assoc vs Triton — final_states max diff: 1.91e-06
Pallas vs Triton — out max diff:          9.54e-07
Pallas vs Triton — final_states max diff: 1.91e-06

All differences should be < 1e-4 for float32 arithmetic


## Autotune BLOCK_SIZE

`dstate` is typically 64–128. We sweep BLOCK_SIZE ∈ {32, 64, 128} to find the fastest tile size.

In [8]:
# Use a representative config for autotuning
_batch, _nchunks, _nheads, _dstate = 1, 32, 128, 128
_key = jax.random.PRNGKey(0)
_states = jax.random.normal(_key, (_batch, _nchunks, _nheads, _dstate), dtype=jnp.float32)
_dA     = jax.random.normal(jax.random.PRNGKey(1), (_batch, _nheads, _nchunks), dtype=jnp.float32) * 0.1

N_WARMUP = 5
N_RUNS   = 20

best_bs, best_ms = None, float('inf')
print(f"Autotuning BLOCK_SIZE for ({_batch}, {_nchunks}, {_nheads}, {_dstate}):")
for bs in [32, 64, 128]:
    if _dstate % bs != 0:
        print(f"  BLOCK_SIZE={bs}: skipped (dstate={_dstate} not divisible)")
        continue
    fn = jax.jit(lambda s, d: state_passing_pallas(s, d, BLOCK_SIZE=bs))
    # warmup
    for _ in range(N_WARMUP):
        out, fin = fn(_states, _dA)
    out.block_until_ready()
    # time
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        out, fin = fn(_states, _dA)
    out.block_until_ready()
    ms = (time.perf_counter() - t0) / N_RUNS * 1e3
    tag = "  <-- best" if ms < best_ms else ""
    print(f"  BLOCK_SIZE={bs:4d}: {ms:.3f} ms{tag}")
    if ms < best_ms:
        best_ms, best_bs = ms, bs

print(f"\nBest BLOCK_SIZE: {best_bs}")
BEST_BLOCK_SIZE = best_bs

Autotuning BLOCK_SIZE for (1, 32, 128, 128):
  BLOCK_SIZE=  32: 0.063 ms  <-- best
  BLOCK_SIZE=  64: 0.079 ms
  BLOCK_SIZE= 128: 0.063 ms

Best BLOCK_SIZE: 32


## Benchmarks

We measure two types of timing:

1. **Standalone**: wall-clock time including JAX Python dispatch overhead (~0.5–1 ms).
   Relevant for code that calls the kernel once per forward pass.

2. **Amortized** (inside `jax.lax.fori_loop`): pure GPU execution time, eliminates Python overhead.
   This gives the true kernel throughput.

### Config legend

| Name | batch | nchunks | nheads | dstate |
|---|---|---|---|---|
| tiny | 1 | 8 | 32 | 64 |
| standard | 1 | 32 | 128 | 128 |
| large_chunks | 1 | 128 | 128 | 64 |
| batched | 4 | 32 | 128 | 128 |

In [9]:
SHAPE_SWEEP = [
    # (batch, nchunks, nheads, dstate),  label
    ((1,   8, 32,  64),  "tiny      B=1 C=8   H=32  D=64"),
    ((1,  32, 128, 128), "standard  B=1 C=32  H=128 D=128"),
    ((1, 128, 128,  64), "lg_chunks B=1 C=128 H=128 D=64"),
    ((4,  32, 128, 128), "batched   B=4 C=32  H=128 D=128"),
]

N_WARMUP   = 10
N_RUNS     = 50
N_AMORTIZE = 200   # fori_loop iterations for amortized timing

def bench_standalone(fn, inputs, label, warmup=N_WARMUP, runs=N_RUNS):
    """Wall-clock timing (includes JAX dispatch overhead)."""
    outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    for _ in range(warmup):
        outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs):
        outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    return (time.perf_counter() - t0) / runs * 1e3

def bench_triton_standalone(states_torch, dA_torch, warmup=N_WARMUP, runs=N_RUNS):
    """Triton standalone timing."""
    for _ in range(warmup):
        out, fin = _state_passing_fwd(states_torch, dA_torch)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        out, fin = _state_passing_fwd(states_torch, dA_torch)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1e3

print("Benchmark utilities defined")

Benchmark utilities defined


In [10]:
print("=" * 78)
print(f"{'Config':<42} {'Triton':>8} {'Naive':>8} {'Assoc':>8} {'Pallas':>8}")
print(f"{'':42} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("-" * 78)

for (batch, nchunks, nheads, dstate), label in SHAPE_SWEEP:
    key = jax.random.PRNGKey(42)
    states_jax = jax.random.normal(key, (batch, nchunks, nheads, dstate), dtype=jnp.float32)
    dA_jax     = jax.random.normal(jax.random.PRNGKey(1),
                                    (batch, nheads, nchunks), dtype=jnp.float32) * 0.1

    # Triton
    s_torch = torch.from_numpy(np.array(states_jax)).cuda().float()
    d_torch = torch.from_numpy(np.array(dA_jax)).cuda().float()
    t_triton = bench_triton_standalone(s_torch, d_torch)

    # Naive JAX
    f_naive = jax.jit(state_passing_naive)
    t_naive = bench_standalone(f_naive, (states_jax, dA_jax), label)

    # Associative scan
    f_assoc = jax.jit(state_passing_assoc)
    t_assoc = bench_standalone(f_assoc, (states_jax, dA_jax), label)

    # Pallas
    bs = BEST_BLOCK_SIZE if dstate % BEST_BLOCK_SIZE == 0 else 64
    f_pallas = jax.jit(lambda s, d: state_passing_pallas(s, d, BLOCK_SIZE=bs))
    t_pallas = bench_standalone(f_pallas, (states_jax, dA_jax), label)

    print(f"{label:<42} {t_triton:>8.3f} {t_naive:>8.3f} {t_assoc:>8.3f} {t_pallas:>8.3f}")

print("=" * 78)
print("\nNote: includes JAX Python dispatch overhead (~0.5-1ms per call)")

Config                                       Triton    Naive    Assoc   Pallas
                                               (ms)     (ms)     (ms)     (ms)
------------------------------------------------------------------------------
tiny      B=1 C=8   H=32  D=64                0.071    0.345    0.050    0.080
standard  B=1 C=32  H=128 D=128               0.084    1.183    0.050    0.052
lg_chunks B=1 C=128 H=128 D=64                0.091    4.730    0.063    0.051
batched   B=4 C=32  H=128 D=128               0.058    1.055    0.050    0.053

Note: includes JAX Python dispatch overhead (~0.5-1ms per call)


In [11]:
# ---- Amortized benchmark via fori_loop ----
# Wrap each function in jax.lax.fori_loop inside jit to eliminate Python dispatch overhead.
# This gives true GPU execution time without Python overhead.

def make_amortized_naive(n_iters):
    def fn(states, dA):
        def body(i, carry):
            out, fin = state_passing_naive(carry[0], dA)
            return (out, fin)
        out0 = jnp.zeros_like(states)
        fin0 = jnp.zeros((states.shape[0], states.shape[2], states.shape[3]), dtype=jnp.float32)
        return lax.fori_loop(0, n_iters, body, (out0, fin0))
    return jax.jit(fn)

def make_amortized_assoc(n_iters):
    def fn(states, dA):
        def body(i, carry):
            out, fin = state_passing_assoc(carry[0], dA)
            return (out, fin)
        out0 = jnp.zeros_like(states)
        fin0 = jnp.zeros((states.shape[0], states.shape[2], states.shape[3]), dtype=jnp.float32)
        return lax.fori_loop(0, n_iters, body, (out0, fin0))
    return jax.jit(fn)

def make_amortized_pallas(n_iters, BLOCK_SIZE):
    def fn(states, dA):
        def body(i, carry):
            out, fin = state_passing_pallas(carry[0], dA, BLOCK_SIZE=BLOCK_SIZE)
            return (out, fin)
        out0 = jnp.zeros_like(states)
        fin0 = jnp.zeros((states.shape[0], states.shape[2], states.shape[3]), dtype=jnp.float32)
        return lax.fori_loop(0, n_iters, body, (out0, fin0))
    return jax.jit(fn)

def bench_amortized(fn, inputs, n_iters, warmup=3, runs=5):
    """Amortized timing: fori_loop over n_iters, divide by n_iters."""
    outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    for _ in range(warmup):
        outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs):
        outs = fn(*inputs)
    if isinstance(outs, (list, tuple)):
        outs[0].block_until_ready()
    else:
        outs.block_until_ready()
    total_s = (time.perf_counter() - t0) / runs
    return total_s / n_iters * 1e3  # ms per iteration


print("=" * 78)
print(f"AMORTIZED (fori_loop N={N_AMORTIZE}, true GPU time)")
print(f"{'Config':<42} {'Triton':>8} {'Naive':>8} {'Assoc':>8} {'Pallas':>8}")
print(f"{'':42} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("-" * 78)

for (batch, nchunks, nheads, dstate), label in SHAPE_SWEEP:
    key = jax.random.PRNGKey(42)
    states_jax = jax.random.normal(key, (batch, nchunks, nheads, dstate), dtype=jnp.float32)
    dA_jax     = jax.random.normal(jax.random.PRNGKey(1),
                                    (batch, nheads, nchunks), dtype=jnp.float32) * 0.1

    # Triton — use torch profiler for fair amortized timing
    s_torch = torch.from_numpy(np.array(states_jax)).cuda().float()
    d_torch = torch.from_numpy(np.array(dA_jax)).cuda().float()
    for _ in range(10):
        _state_passing_fwd(s_torch, d_torch)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_AMORTIZE):
        _state_passing_fwd(s_torch, d_torch)
    torch.cuda.synchronize()
    t_triton = (time.perf_counter() - t0) / N_AMORTIZE * 1e3

    # Naive JAX (amortized)
    f_naive_am = make_amortized_naive(N_AMORTIZE)
    t_naive = bench_amortized(f_naive_am, (states_jax, dA_jax), N_AMORTIZE)

    # Associative scan (amortized)
    f_assoc_am = make_amortized_assoc(N_AMORTIZE)
    t_assoc = bench_amortized(f_assoc_am, (states_jax, dA_jax), N_AMORTIZE)

    # Pallas (amortized)
    bs = BEST_BLOCK_SIZE if dstate % BEST_BLOCK_SIZE == 0 else 64
    f_pallas_am = make_amortized_pallas(N_AMORTIZE, bs)
    t_pallas = bench_amortized(f_pallas_am, (states_jax, dA_jax), N_AMORTIZE)

    print(f"{label:<42} {t_triton:>8.3f} {t_naive:>8.3f} {t_assoc:>8.3f} {t_pallas:>8.3f}")

print("=" * 78)
print("\nNote: amortized numbers exclude Python dispatch overhead.")

AMORTIZED (fori_loop N=200, true GPU time)
Config                                       Triton    Naive    Assoc   Pallas
                                               (ms)     (ms)     (ms)     (ms)
------------------------------------------------------------------------------
tiny      B=1 C=8   H=32  D=64                0.057    0.254    0.054    0.023
standard  B=1 C=32  H=128 D=128               0.103    1.079    0.030    0.036
lg_chunks B=1 C=128 H=128 D=64                0.054    3.701    0.068    0.063
batched   B=4 C=32  H=128 D=128               0.108    1.010    0.061    0.037

Note: amortized numbers exclude Python dispatch overhead.


## Analysis

### Why state_passing is trivially parallelized vs. chunk_state

`chunk_state_fwd` is a GEMM: `states[c] = X.T @ B_scaled`, embarrassingly parallel in batch/head/chunk.

`state_passing_fwd` is a **sequential scan**: each output depends on the previous one.
The only parallelism available is across `(batch * nheads)` and along `dstate` tiles.

For Nemotron (batch=1, nheads=128, dstate=64, BLOCK_SIZE=64): only **128 GPU threads** run in parallel,
each doing `nchunks` sequential load-multiply-add-store iterations. On modern GPUs with thousands of CUDA
cores, this barely fills the hardware. The kernel is **latency-bound**, not throughput-bound.

### Pallas vs Triton

Both compile to an equivalent GPU kernel: `nchunks` iterations of scalar exp + vector FMA.
Performance should be nearly identical for matching BLOCK_SIZE.

Differences arise from:
- **Pallas loop unrolling**: Python `for c in range(nchunks)` generates `nchunks` copies of the kernel body in SASS. For nchunks=128, this is a large kernel — may exceed instruction cache and hurt performance.
- **Triton's loop**: Not unrolled (it's a runtime loop), so instruction cache pressure is lower for large nchunks.
- **BLOCK_SIZE autotune**: Triton autotuned config may differ from ours.

### Associative scan vs sequential

For small nchunks (≤ 32), the O(log n) scan has more overhead than a straight sequential pass.
For large nchunks (128+), associative scan wins if the GPU has enough parallelism to exploit it.
In practice, state_passing is rarely the bottleneck vs. chunk_state (which involves GEMM).

### Standalone vs amortized

The ~0.5–1 ms JAX dispatch overhead is significant for fast kernels. Amortized numbers reflect
true GPU throughput and should be used when evaluating kernel quality.